# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 个人GitHub项目说明

- 每名学生独立完成本Notebook；
- 输入文件固定为`data/E Commerce Dataset.xlsx`；
- 输出固定写入`output/day04_project/`；
- 不要提交教师演示Notebook或教师参考答案；
- 完成后重启内核并从头运行，再推送到个人GitHub仓库。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


def find_project_root(start=None):
    """从当前目录向上寻找种子项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "E Commerce Dataset.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到data/E Commerce Dataset.xlsx")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "E Commerce Dataset.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：/mnt/data/muc-commerce-2-24012412_full/data/E Commerce Dataset.xlsx
项目输出目录：/mnt/data/muc-commerce-2-24012412_full/output/day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 答案：
# 1. 每条记录代表一位电商用户的完整用户画像、使用行为、消费信息以及是否流失(Churn)的单条用户样本
# 2. 目标变量是 Churn（用户是否流失：1=流失，0=留存）
# 3. CustomerID是用户唯一标识ID，属于分类主键，不具备连续数值数学含义，不能做均值、回归等连续数值运算

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    report = pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isnull().sum(),
        "缺失比例(%)": (data.isnull().sum() / len(data) * 100).round(2),
        "唯一值数量": data.nunique()
    })
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# 完成项目初始审计
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df["CustomerID"].duplicated().sum())
print("\nChurn 频数：")
print(raw_df["Churn"].value_counts())
churn_rate = raw_df["Churn"].mean()
print("流失率：", f"{churn_rate:.2%}")

print("\n主要类别字段频数：")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0

Churn 频数：
Churn
0    4682
1     948
Name: count, dtype: int64
流失率： 16.84%

主要类别字段频数：

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    logs = []
    df = data.copy()
    original_rows = len(df)

    # 1. 删除完全重复行
    dup_count = df.duplicated().sum()
    df = df.drop_duplicates()
    logs.append({
        "步骤": "删除完全重复行",
        "处理规则": "移除完全一致的重复记录",
        "处理前行数": original_rows,
        "处理后行数": len(df),
        "影响行数": dup_count
    })

    # 2. 数值字段中位数填充缺失值
    for col in NUMERIC_MISSING_COLS:
        missing_before = df[col].isna().sum()
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        logs.append({
            "步骤": f"填充缺失值: {col}",
            "处理规则": "中位数填充，不使用Churn分组",
            "处理前行数": len(df),
            "处理后行数": len(df),
            "影响行数": missing_before
        })

    # 3. 类别字段标准化映射
    for col, mapping in CATEGORY_MAPPINGS.items():
        for old_val, new_val in mapping.items():
            match_count = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            logs.append({
                "步骤": f"类别映射: {col} {old_val} → {new_val}",
                "处理规则": "统一同类别命名",
                "处理前行数": len(df),
                "处理后行数": len(df),
                "影响行数": match_count
            })

    # 4. 类型转换
    df["Churn"] = df["Churn"].astype(int)
    df["Complain"] = df["Complain"].astype(int)

    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [7]:
# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()

,步骤,处理规则,处理前行数,处理后行数,影响行数
0,删除完全重复行,移除完全一致的重复记录,5630,5630,0
1,填充缺失值: Tenure,中位数填充，不使用Churn分组,5630,5630,264
2,填充缺失值: WarehouseToHome,中位数填充，不使用Churn分组,5630,5630,251
3,填充缺失值: HourSpendOnApp,中位数填充，不使用Churn分组,5630,5630,255
4,填充缺失值: OrderAmountHikeFromlastYear,中位数填充，不使用Churn分组,5630,5630,265
5,填充缺失值: CouponUsed,中位数填充，不使用Churn分组,5630,5630,256
6,填充缺失值: OrderCount,中位数填充，不使用Churn分组,5630,5630,258
7,填充缺失值: DaySinceLastOrder,中位数填充，不使用Churn分组,5630,5630,307
8,类别映射: PreferredLoginDevice Phone → Mobile Phone,统一同类别命名,5630,5630,1231
9,类别映射: PreferredPaymentMode COD → Cash on Delivery,统一同类别命名,5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
import numpy as np

def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }


tenure_bins = [0, 6, 12, 24, 36, np.inf]
tenure_labels = ["0-6个月", "7-12个月", "13-24个月", "25-36个月", "36个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=False)


cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# 3. 异常值报告
check_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = pd.DataFrame([iqr_outlier_summary(cleaned_df[col]) for col in check_cols], index=check_cols)
display(outlier_report)

# 验证 TenureGroup 缺失值
print("TenureGroup缺失行数：", cleaned_df["TenureGroup"].isna().sum())

,Q1,Q3,下限,上限,候选异常值数量
WarehouseToHome,9.00,20.00,-7.50,36.50,2
OrderCount,1.00,3.00,-2.00,6.00,703
CashbackAmount,145.77,196.39,69.84,272.33,438


TenureGroup缺失行数： 0


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# 完成业务规则检查
rule_check = {
    "使用时长小于0": (cleaned_df["Tenure"] < 0).sum(),
    "仓库距离小于0": (cleaned_df["WarehouseToHome"] < 0).sum(),
    "订单数<=0": (cleaned_df["OrderCount"] <= 0).sum(),
    "返现金额小于0": (cleaned_df["CashbackAmount"] < 0).sum()
}
business_rule_report = pd.DataFrame(list(rule_check.items()), columns=["规则", "不合规记录数"])
display(business_rule_report)

# 处理结论：本次检查无负数违规记录，无需额外删除行；若后续出现负数需与业务确认是否为脏数据

,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数<=0,0
3,返现金额小于0,0


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
# 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 基础校验
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique()
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique()
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique()
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns)

# 导出交付文件
quality_before.reset_index(names="字段").to_csv(OUTPUT_DIR / "data_quality_before.csv", index=False, encoding="utf-8-sig")
quality_after.reset_index(names="字段").to_csv(OUTPUT_DIR / "data_quality_after.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
outlier_report.reset_index(names="字段").to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")

print("✅ Day04 全部交付文件导出完成！")
print(f"清洗后数据路径：{OUTPUT_DIR / 'ecommerce_customer_cleaned.csv'}")

✅ Day04 全部交付文件导出完成！
清洗后数据路径：/mnt/data/muc-commerce-2-24012412_full/output/day04_project/ecommerce_customer_cleaned.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？数据质量问题：存在部分数值字段缺失、类别字段命名不统一（Phone、COD、CC、Mobile 别名），存在 IQR 统计识别的候选异常值，无重复行、无负数违规脏数据。
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？处理策略：数值缺失字段用整体中位数填充（不按 Churn 分组填充）；统一类别字段命名；仅记录 IQR 候选异常值，不直接删除；无重复行无需去重。
3. 为什么清洗后的数据可以作为第五天分析的输入？可用于后续分析：缺失值问题已修复、字段格式统一、数据处理过程可追溯、新增分层字段，满足建模分析基础格式。
4. 哪些处理规则仍需要业务人员确认？待确认事项：IQR 识别的极端订单量、返现金额等候选异常数据，需要业务人员确认真实性，不可盲目删除。

## GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] `output/day04_project/`包含清洗后数据、质量报告、清洗日志和异常/业务规则报告；
- [ ] 原始Excel没有被覆盖；
- [ ] 清洗函数、处理日志和项目复盘均已完成；
- [ ] 已提交并推送到个人GitHub仓库。